In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Lat-Height monthly anomalies (hindcast − multi-year climatology)
----------------------------------------------------------------
Processing logic:
1. Hindcast run:
   - Read O3, T, U from hindcast restart experiments.
   - Zonal mean, member mean, -> monthly mean (plev,lat,month).
2. Longrun climatology:
   - Read same vars from long integration MERGED_FILE.
   - Zonal mean, member mean, -> multi-year monthly climatology (plev,lat,month).
3. Month-align (Feb->Feb, etc.) and compute anomalies:
   - O3: convert both hindcast and climatology to DU/km BEFORE subtraction.
     DU/km = 26.956 * P(hPa)/T(K) * O3_ppmv
     where O3_ppmv = O3_vmr * 1e6.
   - T anomaly in K.
   - U anomaly in m/s.

Plotting:
- pressure–latitude sections (pressure is log scale, shown in hPa).
- subpanels: each requested month (Feb init → 2,3,4,5; Mar init → 3,4,5,6).
- publication-style aesthetics:
  * unified Arial-like font, small ticks outward
  * panel labels (a), (b), ...
  * horizontal shared colorbar with units
  * symmetric diverging color map
  * fixed color scale [-10, 10] for all panels in that figure
  * y-axis labeled in hPa
"""

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os, math, glob
from typing import List, Dict, Tuple, Union

# ----------------------------------------------------
# User config / paths
# ----------------------------------------------------
MERGED_FILE = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'
OUTPUT_DIR  = '/home/weiji/restart_exam/plots/LatHeight_Profiles/'
VARIABLES   = ['O3', 'T', 'U']
N_LEVELS    = 21   # odd so 0 sits in center of colormap

MONTH_NAMES = {
    1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
    7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'
}

# Hindcast patterns (extend if you have Apr/May/Jun ...)
file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Feb_nocpl = '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'
file_0008_Mar_nocpl = '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'

hindcast_runs: List[Dict] = [
    {
        'label': '0008 small pertlim (Feb init)',
        'init': 'Feb',
        'year': '0008',
        'files': [file_0008_Feb_small, file_0008_Mar_small]  # add Apr/May dirs if you have them
    },
    {
        'label': '0008 nocouple (Feb init)',
        'init': 'Feb',
        'year': '0008',
        'files': [file_0008_Feb_nocpl, file_0008_Mar_nocpl]
    },
    {
        'label': '0008 small pertlim (Mar init)',
        'init': 'Mar',
        'year': '0008',
        'files': [file_0008_Mar_small]  # extend if Apr/May/Jun available
    },
    {
        'label': '0008 nocouple (Mar init)',
        'init': 'Mar',
        'year': '0008',
        'files': [file_0008_Mar_nocpl]
    },
]

# ----------------------------------------------------
# Plot style for "Nature Geoscience"-like look
# ----------------------------------------------------
def set_plot_style():
    """
    Global matplotlib rcParams for publication-quality styling.
    """
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "xtick.major.width": 0.8,
        "ytick.major.size": 3,
        "ytick.major.width": 0.8,
        "savefig.dpi": 300,
        "figure.dpi": 300,
        "figure.facecolor": "white",
        "text.color": "black",
        "axes.edgecolor": "black",
        "xtick.color": "black",
        "ytick.color": "black",
    })

# ----------------------------------------------------
# Helper functions (data prep)
# ----------------------------------------------------
def months_for_init(init: str) -> List[int]:
    """Feb init -> [2,3,4,5] ; Mar init -> [3,4,5,6]."""
    if init.lower().startswith('feb'):
        return [2,3,4,5]
    elif init.lower().startswith('mar'):
        return [3,4,5,6]
    else:
        raise ValueError("init must be 'Feb' or 'Mar'")

def auto_subplot_layout(n: int) -> Tuple[int,int]:
    # 1 row up to 4 panels looks nice for publication
    if n <= 4:
        return (1, n)
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    return rows, cols

def _open_many(patterns: Union[str, List[str]]) -> xr.Dataset:
    """
    Expand all globs to actual file list and open.
    """
    if isinstance(patterns, str):
        paths = sorted(glob.glob(patterns))
    else:
        paths = []
        for p in patterns:
            paths.extend(glob.glob(p))
        paths = sorted(paths)

    if not paths:
        raise FileNotFoundError(f"No files matched: {patterns}")

    try:
        ds = xr.open_mfdataset(paths, combine='by_coords')
    except Exception:
        ds = xr.open_mfdataset(paths, concat_dim='member', combine='nested')
    return ds

def process_var_latheight(ds: xr.Dataset, var: str) -> xr.DataArray:
    if var not in ds:
        raise KeyError(f"Variable {var} not found in dataset.")
    da = ds[var].mean(dim='lon', keep_attrs=True)
    if 'member' in da.dims:
        da = da.mean(dim='member', keep_attrs=True)
    wanted = [d for d in ['plev','lat','time'] if d in da.dims]
    da = da.transpose(*wanted)

    # === NEW: vertical-level debug ===
    if 'plev' in da.dims:
        p = da['plev'].values
        plevel_unit = da['plev'].attrs.get('units', 'unknown')
        print(f"[Z-DEBUG] var={var}  vertical levels = {p.size}  (plev units: {plevel_unit})")
        print(f"[Z-DEBUG] var={var}  plev min={np.nanmin(p):.3g}, max={np.nanmax(p):.3g}")
        # 显示头尾各 8 个层值，便于人工核对是否规则等距
        head = ", ".join([f"{v:.3g}" for v in p[:min(8, p.size)]])
        tail = ", ".join([f"{v:.3g}" for v in p[max(0, p.size-8):]])
        print(f"[Z-DEBUG] var={var}  plev head: [{head}] ... tail: [{tail}]")
        # 相邻差分的简单统计（看层间距是否规则）
        if p.size > 2:
            dp = np.diff(p)
            print(f"[Z-DEBUG] var={var}  Δplev mean={np.nanmean(dp):.3g}, std={np.nanstd(dp):.3g}  (first 5: {', '.join([f'{d:.3g}' for d in dp[:5]])})")
    else:
        print(f"[Z-DEBUG] var={var}  has no 'plev' dimension")

    return da


def guess_units_label(varname: str, da: xr.DataArray) -> str:
    u = da.attrs.get('units', '')
    if u:
        return u
    if varname == 'U':
        return 'm/s'
    if varname == 'T':
        return 'K'
    if varname == 'O3':
        return 'DU/km'
    return '(anomaly units)'

# ----------------------------------------------------
# O3 DU/km conversion
# ----------------------------------------------------
def o3_monthly_to_du_per_km(o3_mon: xr.DataArray, T_mon: xr.DataArray) -> xr.DataArray:
    """
    Convert monthly-mean O3 mixing ratio (vmr) + T(K) at each (plev,lat,month)
    into DU/km, following:
        DU/km = 26.956 * [ P(hPa) / T(K) ] * O3_ppmv
    O3_ppmv = O3 * 1e6 if O3 is mol/mol (vmr).
    plev is assumed in Pa -> hPa.
    """
    p_hPa = (o3_mon['plev'] / 100.0)    # Pa -> hPa
    o3_ppmv = o3_mon * 1e6              # mol/mol -> ppmv
    COEFF = 26.95560296256018
    dukm = COEFF * p_hPa * o3_ppmv / T_mon
    dukm = dukm.transpose('plev','lat','month')
    dukm.attrs['units'] = 'DU/km'
    dukm.attrs['long_name'] = 'O3 concentration (per km column)'
    return dukm

# ----------------------------------------------------
# Read climatology (multi-year) and hindcast (monthly means)
# ----------------------------------------------------
def read_longrun_climatology_monthly_all(var_list: List[str]) -> Dict[str, xr.DataArray]:
    ds = xr.open_dataset(MERGED_FILE)
    clim_all: Dict[str, xr.DataArray] = {}
    for v in var_list:
        da = process_var_latheight(ds, v)
        raw_units = da.attrs.get('units', 'N/A')
        print(f"[CLIM DEBUG] var={v} original units='{raw_units}' raw shape={da.shape} dims={da.dims}")
        clim_mon = da.groupby('time.month').mean('time', keep_attrs=True).transpose('plev','lat','month')
        clim_mon.attrs['units'] = raw_units
        print(f"[CLIM DEBUG] var={v} climatology shape={clim_mon.shape} months={clim_mon['month'].values.tolist()}")
        clim_all[v] = clim_mon

    # === NEW: cross-variable plev一致性检查 ===
    common = None
    for v, da in clim_all.items():
        p = tuple(np.asarray(da['plev'].values).tolist())
        common = p if common is None else common
        equal = (p == common)
        print(f"[Z-DEBUG] CLIM plev equal to first? var={v}: {equal}")
    ds.close()
    return clim_all

def read_hindcast_monthly_all(files: Union[str, List[str]], var_list: List[str]) -> Dict[str, xr.DataArray]:
    ds = _open_many(files)
    hc_all: Dict[str, xr.DataArray] = {}
    tvals_global = None
    months_here_global = None

    for v in var_list:
        da = process_var_latheight(ds, v)
        units_v = da.attrs.get('units', 'N/A')
        print(f"[HIN DEBUG] var={v} units='{units_v}' raw shape={da.shape} dims={da.dims}")

        if 'time' in da.dims:
            tvals = da['time'].values
            months_here = da['time'].dt.month.values
            unique_months = np.unique(months_here)
            print(f"[HIN DEBUG] var={v} raw hindcast time range: {str(tvals[0])} to {str(tvals[-1])}")
            print(f"[HIN DEBUG] var={v} raw hindcast months: {unique_months.tolist()}")
            for m in unique_months:
                count_m = int(np.sum(months_here == m))
                print(f"[HIN DEBUG] var={v} month {m} has {count_m} timesteps in hindcast")
            tvals_global = tvals
            months_here_global = unique_months

        hc_mon = da.groupby('time.month').mean('time', keep_attrs=True)
        hc_mon = hc_mon.transpose('plev','lat','month')
        hc_mon.attrs['units'] = units_v

        for m in hc_mon['month'].values.tolist():
            field = hc_mon.sel(month=m).values
            print(
                f"[HIN DEBUG] var={v} month {m} hc_mon stats: "
                f"mean={np.nanmean(field):.4g}, std={np.nanstd(field):.4g}, "
                f"min={np.nanmin(field):.4g}, max={np.nanmax(field):.4g}"
            )

        hc_all[v] = hc_mon
    # === NEW: cross-variable plev一致性检查 ===
    common = None
    for v, da in hc_all.items():
        p = tuple(np.asarray(da['plev'].values).tolist())
        common = p if common is None else common
        equal = (p == common)
        print(f"[Z-DEBUG] HIN  plev equal to first? var={v}: {equal}")
    ds.close()

    if tvals_global is not None:
        print(f"[HIN DEBUG] overall time range: {str(tvals_global[0])} to {str(tvals_global[-1])}")
        print(f"[HIN DEBUG] overall unique months: {months_here_global.tolist()}")

    return hc_all

# ----------------------------------------------------
# Anomaly calcs
# ----------------------------------------------------
def compute_anomaly_std(
    hc_mon: xr.DataArray,
    clim_mon: xr.DataArray,
    months_req: List[int],
    var: str
) -> Tuple[xr.DataArray, List[int], str]:
    """
    For T and U:
    anomaly = hindcast_monthly - clim_monthly  (same units)
    """
    hc_have   = hc_mon['month'].values.tolist()
    clim_have = clim_mon['month'].values.tolist()
    used      = sorted(set(months_req).intersection(hc_have).intersection(clim_have))

    if not used:
        raise RuntimeError(
            f"[ERROR] {var}: No overlapping months. "
            f"requested={months_req}, hc_have={hc_have}, clim_have={clim_have}"
        )

    hc_lat   = hc_mon['lat'].values
    cl_lat   = clim_mon['lat'].values
    hc_plev  = hc_mon['plev'].values
    cl_plev  = clim_mon['plev'].values
    same_lat  = np.array_equal(hc_lat, cl_lat)
    same_plev = np.array_equal(hc_plev, cl_plev)

    print(f"[ALIGN DEBUG] var={var} requested={months_req}, hc_have={hc_have}, clim_have={clim_have}, used={used}")
    print(f"[ALIGN DEBUG] var={var} hc_mon shape={hc_mon.shape}, clim_mon shape={clim_mon.shape}")
    print(f"[ALIGN DEBUG] var={var} lat equal? {same_lat}, hc_lat.shape={hc_lat.shape}, clim_lat.shape={cl_lat.shape}")
    print(f"[ALIGN DEBUG] var={var} plev equal? {same_plev}, hc_plev.shape={hc_plev.shape}, clim_plev.shape={cl_plev.shape}")

    anom = hc_mon.sel(month=used) - clim_mon.sel(month=used)

    for m in used:
        clim_field = clim_mon.sel(month=m).values
        anom_field = anom.sel(month=m).values
        print(
            f"[CLIM DEBUG] var={var} month {m} clim stats: "
            f"mean={np.nanmean(clim_field):.4g}, std={np.nanstd(clim_field):.4g}, "
            f"min={np.nanmin(clim_field):.4g}, max={np.nanmax(clim_field):.4g}"
        )
        print(
            f"[ANOM DEBUG] var={var} month {m} anomaly stats: "
            f"mean={np.nanmean(anom_field):.4g}, std={np.nanstd(anom_field):.4g}, "
            f"max|.|={np.nanmax(np.abs(anom_field)):.4g}"
        )
        if m == 2:
            print(
                f"[ANOM DEBUG FEB CHECK] var={var} Feb anomaly stats again: "
                f"mean={np.nanmean(anom_field):.4g}, std={np.nanstd(anom_field):.4g}, "
                f"max|.|={np.nanmax(np.abs(anom_field)):.4g}"
            )

    anom = anom.transpose('plev','lat','month')
    units_label = guess_units_label(var, hc_mon)
    return anom, used, units_label

def compute_anomaly_o3_dukm(
    hc_all: Dict[str, xr.DataArray],
    clim_all: Dict[str, xr.DataArray],
    months_req: List[int]
) -> Tuple[xr.DataArray, List[int], str]:
    """
    For O3:
    1. Convert hindcast monthly (O3,T) -> DU/km
    2. Convert climatology monthly (O3,T) -> DU/km
    3. Subtract => anomaly in DU/km
    """
    hc_o3_mon = hc_all['O3']
    hc_t_mon  = hc_all['T']
    cl_o3_mon = clim_all['O3']
    cl_t_mon  = clim_all['T']

    print(f"[O3 DEBUG] hindcast O3 units='{hc_o3_mon.attrs.get('units','N/A')}', T='{hc_t_mon.attrs.get('units','N/A')}'")
    print(f"[O3 DEBUG] clim O3 units='{cl_o3_mon.attrs.get('units','N/A')}',     T='{cl_t_mon.attrs.get('units','N/A')}'")
    print("[O3 DEBUG] Assuming O3 is vmr (dimensionless mixing ratio).")

    # convert both to DU/km
    print("[O3 DEBUG] Converting hindcast O3,T to DU/km ...")
    hc_dukm = o3_monthly_to_du_per_km(hc_o3_mon, hc_t_mon)
    for m in hc_dukm['month'].values.tolist():
        arr = hc_dukm.sel(month=m).values
        print(f"[O3 DEBUG] hindcast DU/km month {m}: mean={np.nanmean(arr):.4g}, "
              f"std={np.nanstd(arr):.4g}, min={np.nanmin(arr):.4g}, max={np.nanmax(arr):.4g}")

    print("[O3 DEBUG] Converting clim O3,T to DU/km ...")
    cl_dukm = o3_monthly_to_du_per_km(cl_o3_mon, cl_t_mon)
    for m in cl_dukm['month'].values.tolist():
        arr = cl_dukm.sel(month=m).values
        print(f"[O3 DEBUG] clim DU/km month {m}: mean={np.nanmean(arr):.4g}, "
              f"std={np.nanstd(arr):.4g}, min={np.nanmin(arr):.4g}, max={np.nanmax(arr):.4g}")

    hc_have   = hc_dukm['month'].values.tolist()
    clim_have = cl_dukm['month'].values.tolist()
    used      = sorted(set(months_req).intersection(hc_have).intersection(clim_have))
    if not used:
        raise RuntimeError(
            f"[ERROR] O3(DU/km): No overlapping months. "
            f"requested={months_req}, hc_have={hc_have}, clim_have={clim_have}"
        )

    same_lat  = np.array_equal(hc_dukm['lat'].values,  cl_dukm['lat'].values)
    same_plev = np.array_equal(hc_dukm['plev'].values, cl_dukm['plev'].values)
    print(f"[ALIGN DEBUG] var=O3(DU/km) requested={months_req}, hc_have={hc_have}, clim_have={clim_have}, used={used}")
    print(f"[ALIGN DEBUG] var=O3(DU/km) hc_dukm shape={hc_dukm.shape}, cl_dukm shape={cl_dukm.shape}")
    print(f"[ALIGN DEBUG] var=O3(DU/km) lat equal? {same_lat}, "
          f"hc_lat.shape={hc_dukm['lat'].shape}, clim_lat.shape={cl_dukm['lat'].shape}")
    print(f"[ALIGN DEBUG] var=O3(DU/km) plev equal? {same_plev}, "
          f"hc_plev.shape={hc_dukm['plev'].shape}, clim_plev.shape={cl_dukm['plev'].shape}")

    anom_dukm = hc_dukm.sel(month=used) - cl_dukm.sel(month=used)

    for m in used:
        clim_field = cl_dukm.sel(month=m).values
        anom_field = anom_dukm.sel(month=m).values
        print(
            f"[CLIM DEBUG] var=O3(DU/km) month {m} clim stats: "
            f"mean={np.nanmean(clim_field):.4g}, std={np.nanstd(clim_field):.4g}, "
            f"min={np.nanmin(clim_field):.4g}, max={np.nanmax(clim_field):.4g}"
        )
        print(
            f"[ANOM DEBUG] var=O3(DU/km) month {m} anomaly stats: "
            f"mean={np.nanmean(anom_field):.4g}, std={np.nanstd(anom_field):.4g}, "
            f"max|.|={np.nanmax(np.abs(anom_field)):.4g}"
        )
        if m == 2:
            print(
                f"[ANOM DEBUG FEB CHECK] var=O3(DU/km) Feb anomaly stats again: "
                f"mean={np.nanmean(anom_field):.4g}, std={np.nanstd(anom_field):.4g}, "
                f"max|.|={np.nanmax(np.abs(anom_field)):.4g}"
            )

    anom_dukm = anom_dukm.transpose('plev','lat','month')
    units_label = "DU/km"
    return anom_dukm, used, units_label

# ----------------------------------------------------
# Plotting (publication style)
# ----------------------------------------------------
def _apply_pressure_ticks(ax, plevels_pa: np.ndarray):
    """
    Put y-axis ticks in hPa, log-pressure inverted.
    """
    cand_hpa = np.array([1000, 700, 500, 300, 200, 100, 70, 50, 30, 20, 10, 5, 3, 1])
    cand_pa  = cand_hpa * 100.0
    pmin = np.nanmin(plevels_pa)
    pmax = np.nanmax(plevels_pa)

    mask = (cand_pa >= pmin) & (cand_pa <= pmax)
    ticks_pa = cand_pa[mask]
    ticklabels = [str(int(v)) for v in cand_hpa[mask]]

    if len(ticks_pa) > 0:
        ax.set_yticks(ticks_pa)
        ax.set_yticklabels(ticklabels)
    ax.set_ylabel('Pressure (hPa)')

def plot_latheight_anomaly(
    anom: xr.DataArray,
    months: List[int],
    var: str,
    run_label: str,
    init_label: str,
    year: str,
    units_label: str,
    save_path: str,
    n_levels: int = N_LEVELS
):
    """
    Multi-panel plot (one panel per requested month):
    - symmetric, FIXED color scale [-10, 10]
    - NO zero-contour overlay anymore
    - panel labels (a,b,c,...)
    - horizontal shared colorbar with units
    - log-pressure y-axis in hPa
    """

    # fixed global color scale per figure
    fixed_vmin = -10.0
    fixed_vmax =  10.0
    levels = np.linspace(fixed_vmin, fixed_vmax, n_levels)

    rows, cols = auto_subplot_layout(len(months))

    # figure size tuned for publication (room for title + colorbar)
    fig_w = 4.8 * cols
    fig_h = 4.0 * rows + 1.2
    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), squeeze=False)

    plevels = anom['plev'].values
    lats    = anom['lat'].values
    panel_letters = list("abcdefghijklmnopqrstuvwxyz")

    cf_last = None

    for i, m in enumerate(months):
        r, c = divmod(i, cols)
        ax = axes[r][c]

        field = anom.sel(month=m).values  # (plev,lat) for that month

        # filled contours only
        cf_last = ax.contourf(
            lats, plevels, field,
            levels=levels,
            cmap='RdBu_r',
            extend='both',
            vmin=fixed_vmin,
            vmax=fixed_vmax,
        )

        # y-axis: log-pressure, inverted
        ax.set_yscale('log')
        ax.invert_yaxis()

        # x-axis: latitude ticks
        ax.set_xlim(lats.min(), lats.max())
        ax.set_xticks(np.arange(-90, 91, 30))
        ax.set_xlabel('Latitude (°)')

        # nice pressure ticks (in hPa)
        _apply_pressure_ticks(ax, plevels)

        # thin spines
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

        # panel title = month name
        ax.set_title(f"{MONTH_NAMES.get(int(m), str(m))}", fontsize=8, pad=2)

        # panel label (a),(b),(c)...
        ax.text(
            0.03, 0.95,
            f"({panel_letters[i]})",
            transform=ax.transAxes,
            ha='left', va='top',
            fontsize=8,
            fontweight='bold',
            color='black'
        )

    # hide unused axes
    for j in range(len(months), rows*cols):
        r, c = divmod(j, cols)
        axes[r][c].axis('off')

    # layout margins
    plt.subplots_adjust(
        left=0.10,
        right=0.90,
        top=0.82,
        bottom=0.22,
        wspace=0.4,
        hspace=0.6
    )

    # shared horizontal colorbar under panels
    if cf_last is not None:
        cbar_ax = fig.add_axes([0.10, 0.10, 0.80, 0.03])
        cb_ticks = np.linspace(fixed_vmin, fixed_vmax, 5)  # [-10,-5,0,5,10]
        cbar = fig.colorbar(
            cf_last,
            cax=cbar_ax,
            orientation='horizontal',
            ticks=cb_ticks
        )
        cbar.set_label(f"{var} anomaly [{units_label}]", fontsize=8)
        for tick in cbar.ax.xaxis.get_ticklines():
            tick.set_linewidth(0.8)

    # figure-level title block
    fig.text(
        0.5, 0.88,
        f"{run_label}, Year {year}",
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )
    fig.text(
        0.5, 0.845,
        f"{var} anomaly [{units_label}]   Init={init_label}",
        ha='center', va='bottom', fontsize=8
    )

    # save
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[INFO] Saved figure: {save_path}")

# ----------------------------------------------------
# Driver
# ----------------------------------------------------
def generate_all_latheight():
    # apply figure style first
    set_plot_style()

    print("[INFO] Loading multi-year climatology (monthly means) for O3,T,U ...")
    clim_all = read_longrun_climatology_monthly_all(VARIABLES)
    print("[INFO] Done loading climatology.\n")

    for run in hindcast_runs:
        run_label  = run.get('label', 'hindcast')
        init_label = run['init']
        year       = run.get('year', 'NA')
        files      = run['files']
        months_req = months_for_init(init_label)

        print("="*80)
        print(f"[RUN START] {run_label} | Init={init_label} | Year={year}")
        print(f"[RUN INFO ] months_req={months_req}")
        print("="*80)

        # hindcast (monthly means)
        hc_all = read_hindcast_monthly_all(files, VARIABLES)
        print("-"*80)

        # O3 anomaly (DU/km)
        print("[STEP] Computing O3 anomaly (DU/km)...")
        anom_o3, months_used_o3, units_o3 = compute_anomaly_o3_dukm(
            hc_all, clim_all, months_req
        )
        print(f"[FINAL DEBUG] var=O3(DU/km) months_used={months_used_o3}")
        print(f"[FINAL DEBUG] var=O3(DU/km) anomaly shape={anom_o3.shape} dims={anom_o3.dims}")
        out_name = f"{run_label.replace(' ', '_').replace('/', '-')}_O3.png"
        save_path = os.path.join(OUTPUT_DIR, out_name)
        plot_latheight_anomaly(
            anom_o3, months_used_o3, 'O3', run_label, init_label, year, units_o3, save_path
        )
        print("-"*80)

        # T anomaly (K)
        print("[STEP] Computing T anomaly...")
        anom_t, months_used_t, units_t = compute_anomaly_std(
            hc_all['T'], clim_all['T'], months_req, 'T'
        )
        print(f"[FINAL DEBUG] var=T months_used={months_used_t}")
        print(f"[FINAL DEBUG] var=T anomaly shape={anom_t.shape} dims={anom_t.dims}")
        out_name = f"{run_label.replace(' ', '_').replace('/', '-')}_T.png"
        save_path = os.path.join(OUTPUT_DIR, out_name)
        plot_latheight_anomaly(
            anom_t, months_used_t, 'T', run_label, init_label, year, units_t, save_path
        )
        print("-"*80)

        # U anomaly (m/s)
        print("[STEP] Computing U anomaly...")
        anom_u, months_used_u, units_u = compute_anomaly_std(
            hc_all['U'], clim_all['U'], months_req, 'U'
        )
        print(f"[FINAL DEBUG] var=U months_used={months_used_u}")
        print(f"[FINAL DEBUG] var=U anomaly shape={anom_u.shape} dims={anom_u.dims}")
        out_name = f"{run_label.replace(' ', '_').replace('/', '-')}_U.png"
        save_path = os.path.join(OUTPUT_DIR, out_name)
        plot_latheight_anomaly(
            anom_u, months_used_u, 'U', run_label, init_label, year, units_u, save_path
        )
        print("="*80 + "\n")

    print("[INFO] All lat-height anomaly figures have been generated.\n")

# ----------------------------------------------------
# main
# ----------------------------------------------------
if __name__ == "__main__":
    generate_all_latheight()
